# Model Preparation : Naive Bayes

Veuillez tester la préparation du modèle sur chaque fichier CSV nettoyé en amont et comparer les résultats obtenus.

### Librairies requises

In [ ]:
# !pip uninstall -y matplotlib
# !pip install matplotlib
!pip install scikit-learn
!pip install pandas # seaborn matplotlib

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
# import matplotlib.pyplot as plt
# import seaborn as sns
import pandas as pd
import joblib, os

## Préparation du modèle

In [ ]:
naive_bayes_model = GaussianNB()

## Préparation de données pour entraînement

In [ ]:
path = input("Entrer le chemin du dataset CSV pour l'entrainement: ")

if not path.endswith('.csv') or not path:
    path = "../data/movies_cleaned.csv"

In [ ]:
def prepare_data(file_path, targeted_feature="genre_encoded", test_size=0.2, random_state=42):
    dataframe = pd.read_csv(file_path)
    X = dataframe.drop(targeted_feature, axis=1)
    y = dataframe[targeted_feature]
    return train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(path, targeted_feature="genre_encoded")

## Entrainement du modèle

In [ ]:
naive_bayes_model.fit(X_train, y_train)

## Test du modèle

In [ ]:
y_pred = naive_bayes_model.predict(X_test)
y_pred

- score() → accuracy (précision) du modèle

In [ ]:
score_bayes = naive_bayes_model.score(X_test, y_test)
score_bayes = round(score_bayes * 100, 2)
print(f"Accuracy: {score_bayes}%")

- f1-score → mesure de la précision du modèle en prenant en compte à la fois la précision et le rappel

In [ ]:
F1_score_bayes = classification_report(y_test, y_pred, output_dict=True)
F1_score_bayes = F1_score_bayes['weighted avg']['f1-score']
print(f"F1-score: {F1_score_bayes:.2f}")

- recall → mesure de la capacité du modèle à identifier correctement les classes positives

In [ ]:
recall_bayes = classification_report(y_test, y_pred, output_dict=True)
recall_bayes = recall_bayes['weighted avg']['recall']
print(f"Recall: {recall_bayes:.2f}")

- matrice de confusion → table pour comparer les prédictions du modèle avec les vraies étiquettes

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :\n", cm)

In [ ]:
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
# disp.plot(cmap=plt.cm.Blues)
# plt.title("Matrice de confusion")
# plt.show()

- Rapport métriques par classe → précision, rappel et F1-score pour chaque classe

In [ ]:
report = classification_report(y_test, y_pred)
report

## Débuggage du modèle

In [ ]:
# A. Train/Test split correct ?
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("y_train value_counts():\n", pd.Series(y_train).value_counts())
print("y_test value_counts():\n", pd.Series(y_test).value_counts())

# B. Baseline naïve = majority class
majority_class = pd.Series(y_train).mode()[0]
baseline_score = (y_test == majority_class).mean()
print(f"Baseline majority class: {baseline_score:.3f}")

In [ ]:
# A. Features nulles ou constantes ?
print("Features variance:")
print(pd.DataFrame(X_train).var().sort_values().head(10))

# B. Features mal scalées ? (pour GaussianNB)
print("Features range:")
print(pd.DataFrame(X_train).describe().loc[['min', 'max']].T)

# C. Features binaires/catégoriques encodées en continu ?
print("Features très concentrées:")
print((pd.DataFrame(X_train).apply(lambda x: len(x.unique())) < 10).sum())

In [ ]:
y_pred = naive_bayes_model.predict(X_test)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
cm
# sns.heatmap(cm, annot=True, fmt='d')
# plt.show()

- test : scaler + modèle

In [ ]:
pipeline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

pipeline_model.fit(X_train, y_train)
print("GaussianNB + Scaler:", pipeline_model.score(X_test, y_test))
score_pipe = pipeline_model.score(X_test, y_test) * 100

- test : dummy classifier (random)

In [ ]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
print("Dummy baseline:", dummy_model.score(X_test, y_test))
score_dummy = dummy_model.score(X_test, y_test) * 100

In [ ]:
print(f"Naive Bayes: {score_bayes:.2f}%")
print(f"Naive Bayes + Scaler: {score_pipe:.2f}%")
print(f"Dummy baseline: {score_dummy:.2f}%")

- Comparison des 3 modèles

In [ ]:
if score_pipe > score_bayes and score_pipe > score_dummy:
    print("Le pipeline avec scaler est le meilleur modèle.")
elif score_bayes > score_pipe and score_bayes > score_dummy:
    print("Le modèle Naive Bayes sans scaler est le meilleur modèle.")
else:
    print("Le baseline Dummy est le meilleur modèle.")
    

## Exportation du modèle

In [ ]:
PATH_OUTPUT_MODEL = "../model"
if not os.path.exists(PATH_OUTPUT_MODEL):
    os.makedirs(PATH_OUTPUT_MODEL, exist_ok=True)

joblib.dump(naive_bayes_model, f"{PATH_OUTPUT_MODEL}/naive_bayes_model.pkl")
joblib.dump(pipeline_model, f"{PATH_OUTPUT_MODEL}/pipeline_model.pkl")
joblib.dump(dummy_model, f"{PATH_OUTPUT_MODEL}/dummy_model.pkl")

# Ressouces 

- [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)